# Uncertainty-aware Composition

A worked example: estimate the **peak gestational cardiac
output** (preconception baseline + peak rise) with explicit
uncertainty propagation, and verify that the derived quantity
inherits the tier of its weakest input.

This notebook demonstrates one of the most useful things you
can do with nidus: combine its citation-backed parameters into
a derived quantity, while keeping uncertainty and provenance
intact.

## Setup

In [ ]:
import nidus
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=20251101)  # deterministic

ds = nidus.load()

## Inputs

Two parameters describe maternal cardiac output across
pregnancy. The dataset stores them as separate records, each
with its own central value, low/high (1σ) bounds, units, tier,
and citation chain.

In [ ]:
baseline = ds["maternal_cardiovascular.baseline_cardiac_output_l_per_min"]
peak_excess = ds["maternal_cardiovascular.peak_excess_cardiac_output_l_per_min"]

for p in (baseline, peak_excess):
    print(f"{p.id}")
    print(f"  value:    {p.value.central} {p.value.units} (low {p.value.low}, high {p.value.high})")
    print(f"  tier:     {p.tier}")
    print(f"  citation: {p.primary_citation.key} (DOI: {p.primary_citation.doi})")
    print()

## Point-estimate calculation

The naïve sum of central values gives a single number with no
information about uncertainty.

In [ ]:
point_peak = baseline.value.central + peak_excess.value.central
print(f"Point estimate of peak gestational CO: {point_peak:.2f} L/min")

## Monte Carlo from low/high bounds

Both source records have ``distribution = "normal"`` and store
their bounds at one standard deviation (``ci = 0.683``). Drawing
Gaussians with the appropriate mean and σ recovers the original
distributions; the sum of two independent Gaussians is itself
Gaussian.

In [ ]:
def sample(p, n):
    assert p.value.distribution == "normal"
    sigma = (p.value.high - p.value.low) / 2  # one-sigma
    return rng.normal(p.value.central, sigma, size=n)

N = 50_000
baseline_samples = sample(baseline, N)
peak_excess_samples = sample(peak_excess, N)
peak_co = baseline_samples + peak_excess_samples

print(f"Mean:  {peak_co.mean():.2f} L/min")
print(f"Std:   {peak_co.std():.2f} L/min")
q025, q975 = np.quantile(peak_co, [0.025, 0.975])
print(f"95% CI: {q025:.2f} – {q975:.2f} L/min")

## Visualise the propagated distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(peak_co, bins=60, alpha=0.7, color="#1f77b4", edgecolor="white")
ax.axvline(point_peak, color="black", linestyle="--", linewidth=1, label=f"point estimate ({point_peak:.2f})")
ax.axvline(q025, color="#d62728", linestyle=":", linewidth=1, label="95% CI bounds")
ax.axvline(q975, color="#d62728", linestyle=":", linewidth=1)
ax.set_xlabel("Peak gestational cardiac output (L/min)")
ax.set_ylabel("Samples")
ax.set_title("Monte Carlo composition: baseline + peak excess")
ax.legend()
plt.tight_layout()
plt.show()

## Tier propagation

Both inputs are Tier B, so the derived quantity is Tier B too.
The rule is conservative: a derived quantity cannot be
better-supported than its weakest input.

In [ ]:
TIER_ORDER = ["A", "B", "C", "D"]
def propagate(*params):
    return max((p.tier for p in params), key=TIER_ORDER.index)

derived_tier = propagate(baseline, peak_excess)
print(f"baseline tier:     {baseline.tier}")
print(f"peak_excess tier:  {peak_excess.tier}")
print(f"derived tier:      {derived_tier}")

## What this means

- **Both inputs are Tier B.** Mahendru 2014 is the primary
  source; multiple cohort studies confirm the qualitative
  shape, but the framework treats the specific values as
  supported-but-not-confirmed.
- **Distributions are one-sigma normal.** This matches how the
  source TOML encoded ``(mean, sd)``. A wider 95% CI
  interpretation would use ``± 1.96σ`` instead.
- **Independence is assumed.** Real CO components are
  correlated (women with higher baselines tend to have larger
  absolute increases). The Monte Carlo assumes independence,
  which slightly overstates the spread.
- **Mahendru 2014 is one cohort.** Replicating across cohorts
  would promote both inputs (and therefore the derived
  quantity) to Tier A.

See the [Spec 03 blog essay outline](../docs/specs/v0.3/03-outreach-and-essay.md)
for the longer narrative that this calculation supports.